# Milestone 1: Data Collection and Preparation

The supplied `healthcare_dataset.csv` contains combined patient-admission and hospital attributes. Hospital and department operational measures are derived from this source; no separate external operational dataset is integrated. Run the pipeline, then review aggregate quality checks below. This notebook intentionally avoids displaying patient-level rows.

In [ ]:
from pathlib import Path
import runpy
import pandas as pd

project_dir = Path.cwd()
required_files = [
    project_dir / "healthcare_dataset.csv",
    project_dir / "data_collection.py",
]
missing_files = [path.name for path in required_files if not path.exists()]
assert not missing_files, f"Missing required files: {missing_files}"
runpy.run_path(str(project_dir / "data_collection.py"))

In [ ]:
source = pd.read_csv(project_dir / "healthcare_dataset.csv")
raw = pd.read_csv(project_dir / "hospital_raw_data.csv")
cleaned = pd.read_csv(project_dir / "hospital_cleaned.csv")
assert len(raw) == len(cleaned), "Integrated and cleaned outputs should contain the same prepared records."

In [ ]:
completeness_pct = 100 * cleaned.notna().to_numpy().sum() / cleaned.size
missing_pct = 100 - completeness_pct
quality_summary = pd.DataFrame([{
    "source_rows": len(source),
    "source_exact_duplicates": int(source.duplicated().sum()),
    "integrated_rows": len(raw),
    "cleaned_exact_duplicates": int(cleaned.duplicated().sum()),
    "cleaned_columns": len(cleaned.columns),
    "completeness_pct": round(completeness_pct, 3),
    "missing_pct": round(missing_pct, 3),
}])
print(quality_summary.to_string(index=False))

In [ ]:
assert (project_dir / "hospital_raw_data.csv").exists()
assert (project_dir / "hospital_cleaned.csv").exists()
assert cleaned.duplicated().sum() == 0
assert completeness_pct > 95, f"Completeness is {completeness_pct:.2f}%"
assert missing_pct < 2, f"Missing values are {missing_pct:.2f}%"
print("Milestone 1 data-quality checks passed.")

In [ ]:
department_counts = (
    cleaned.groupby("department").size()
    .rename("patient_records")
    .reset_index()
)
print(department_counts.to_string(index=False))